# Chapter 9 — Probabilistic Models for Classification I: K-NN, Evaluation Metrics & Reference Templates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch09_probabilistic_models_for_classification_i/ch09_notebook.ipynb)

Companion notebook for **Chapter 9** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Split data into training, validation and test sets. With imbalanced classes, split with stratification first, then balance only the training set.
- K-NN labels a point by a majority vote of its K nearest training examples. Choose K on validation data.
- K-NN is a lazy learner: training is instant, but every prediction compares against all stored examples.
- The confusion matrix gives TP, TN, FP and FN. From these come accuracy, precision, recall and F1.
- Favor precision when false alarms are costly, and recall when misses are costly.
- Reference templates compare a point with each class mean. Mahalanobis distance also accounts for each class's spread.

## Example 1: Nearest Neighbor and K-NN from Scratch

This follows the four steps: compute distances, sort, take the first K, then vote. For the test person (150 cm, 47 kg) the four nearest examples are all adults, at distances 5.39, 8.73, 18.48 and 19.58. For a second person (140 cm, 46 kg), K = 1 and K = 3 say 'Adult' but K = 5 says 'Child'. The choice of K really can change the answer. The last part picks K for the Iris flowers using a validation set.

In [1]:
# Chapter 9 - Example 1: Nearest neighbor and K-NN from scratch (adult vs child)
import numpy as np
from collections import Counter

X = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
              [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5],
              [152, 52.0], [158, 50.5], [161, 63.2], [165, 57.8], [168, 71.0],
              [172, 64.5], [175, 80.3], [179, 72.9], [183, 88.0], [188, 81.4]])
y = np.r_[np.zeros(10, int), np.ones(10, int)]
NAMES = {0: "Child", 1: "Adult"}

# Raw cm and kg are used so the numbers match Figure 9.1. In general, standardize the
# features before K-NN (Chapter 6), as Example 2 does.
def knn_predict(x, K):
    d = np.sqrt(((X - x) ** 2).sum(axis=1))        # Step 1: Euclidean distance to all
    order = np.argsort(d)                           # Step 2: sort, nearest first
    nearest = order[:K]                             # Step 3: take the first K
    vote = Counter(y[nearest]).most_common(1)[0][0] # Step 4: majority vote
    return vote, d, order

for test in ([150, 47.0], [140, 46.0]):
    print(f"Test person: height {test[0]} cm, weight {test[1]} kg")
    _, d, order = knn_predict(np.array(test), 1)
    print("  5 nearest training examples:")
    for i in order[:5]:
        print(f"    ({X[i, 0]:.0f}, {X[i, 1]:5.2f})  distance {d[i]:6.2f}  class {NAMES[y[i]]}")
    for K in (1, 3, 5):
        print(f"  K = {K}: predicted {NAMES[knn_predict(np.array(test), K)[0]]}")
    print()

# Choosing K with a validation set (Iris flowers: 150 samples, 3 classes)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
Xi, yi = load_iris(return_X_y=True)
X_tr, X_val, y_tr, y_val = train_test_split(Xi, yi, test_size=0.4, stratify=yi, random_state=3)
print("Choosing K on a validation set (Iris):")
for K in (1, 3, 5, 7, 9, 15, 25):
    acc = KNeighborsClassifier(n_neighbors=K).fit(X_tr, y_tr).score(X_val, y_val)
    print(f"  K = {K:2d}: validation accuracy {acc:.3f}")

Test person: height 150 cm, weight 47.0 kg
  5 nearest training examples:
    (152, 52.00)  distance   5.39  class Adult
    (158, 50.50)  distance   8.73  class Adult
    (165, 57.80)  distance  18.48  class Adult
    (161, 63.20)  distance  19.58  class Adult
    (134, 31.00)  distance  22.63  class Child
  K = 1: predicted Adult
  K = 3: predicted Adult
  K = 5: predicted Adult

Test person: height 140 cm, weight 46.0 kg
  5 nearest training examples:
    (152, 52.00)  distance  13.42  class Adult
    (134, 31.00)  distance  16.16  class Child
    (158, 50.50)  distance  18.55  class Adult
    (131, 26.30)  distance  21.66  class Child
    (127, 27.10)  distance  22.94  class Child
  K = 1: predicted Adult
  K = 3: predicted Adult
  K = 5: predicted Child

Choosing K on a validation set (Iris):
  K =  1: validation accuracy 0.967
  K =  3: validation accuracy 0.983
  K =  5: validation accuracy 0.967
  K =  7: validation accuracy 0.983
  K =  9: validation accuracy 0.983
  K = 15: v

## Example 2: Confusion Matrix, Precision, Recall and F1

We classify breast tumors as malignant (positive) or benign, using K-NN on a stratified 70/30 split. The program prints the confusion matrix, then works out accuracy, precision, recall and F1 by hand and checks them against scikit-learn. Both agree. Look at the false negatives: 4 cancers were missed. That number is why recall is the metric doctors care about most.

In [2]:
# Chapter 9 - Example 2: Evaluating a classifier - confusion matrix, accuracy,
# precision, recall and F1, with a stratified 70/30 split
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

data = load_breast_cancer()
X, y = data.data, 1 - data.target        # make 1 = malignant (the 'positive' class we must catch)
print(f"Samples: {len(y)}  (malignant {y.sum()}, benign {len(y) - y.sum()})")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"Train {len(y_tr)} / Test {len(y_te)}; malignant share train "
      f"{y_tr.mean():.2f}, test {y_te.mean():.2f} (stratified = same mix)")

sc = StandardScaler().fit(X_tr)
model = KNeighborsClassifier(n_neighbors=5).fit(sc.transform(X_tr), y_tr)
pred = model.predict(sc.transform(X_te))

tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print("\nConfusion matrix          Actual +   Actual -")
print(f"  Predicted + (malignant)   TP={tp:3d}     FP={fp:3d}")
print(f"  Predicted - (benign)      FN={fn:3d}     TN={tn:3d}")

acc = (tp + tn) / (tp + tn + fp + fn)
prec = tp / (tp + fp)
rec = tp / (tp + fn)
f1 = 2 * prec * rec / (prec + rec)
print(f"\nBy hand : accuracy {acc:.3f}  precision {prec:.3f}  recall {rec:.3f}  F1 {f1:.3f}")
print(f"sklearn : precision {precision_score(y_te, pred):.3f}  recall "
      f"{recall_score(y_te, pred):.3f}  F1 {f1_score(y_te, pred):.3f}")
print(f"\n{fn} cancer case(s) were missed (false negatives). In medicine, recall matters most.")

Samples: 569  (malignant 212, benign 357)
Train 398 / Test 171; malignant share train 0.37, test 0.37 (stratified = same mix)

Confusion matrix          Actual +   Actual -
  Predicted + (malignant)   TP= 60     FP=  6
  Predicted - (benign)      FN=  4     TN=101

By hand : accuracy 0.942  precision 0.909  recall 0.938  F1 0.923
sklearn : precision 0.909  recall 0.938  F1 0.923

4 cancer case(s) were missed (false negatives). In medicine, recall matters most.


## Example 3: Reference Templates — Euclidean vs Mahalanobis

Training is just computing each class's mean and covariance: the child mean is [116.2, 21.85] and the adult mean is [170.1, 68.16]. For the first test person both distances agree: Euclidean 42.13 vs 29.18, Mahalanobis 8.13 vs 1.76, so the answer is adult. The second test person (137 cm, 38 kg) shows the two distances disagreeing. In plain distance they are nearer the children (26.33 vs 44.78). But children's weight follows their height very closely, and 38 kg is unusually heavy for a 137 cm child. For adults, 38 kg at 137 cm sits right on the usual height-weight trend. Mahalanobis takes that pattern into account and picks adult (2.94 vs 5.40).

In [3]:
# Chapter 9 - Example 3: Reference templates - Euclidean vs Mahalanobis distance
import numpy as np
np.set_printoptions(suppress=True)

child = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
                  [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5]])
adult = np.array([[152, 52.0], [158, 50.5], [161, 63.2], [165, 57.8], [168, 71.0],
                  [172, 64.5], [175, 80.3], [179, 72.9], [183, 88.0], [188, 81.4]])

# Training = computing statistics of each class
templates = {}
for name, data in (("Child", child), ("Adult", adult)):
    mu = data.mean(axis=0)                    # first-order statistic: the mean
    cov = np.cov(data, rowvar=False)          # second-order: covariance (divides by N-1)
    templates[name] = (mu, cov)
    print(f"{name}: mean = {np.round(mu, 2)}, covariance =\n{np.round(cov, 2)}\n")

x = np.array([150, 47.0])
print(f"Test person x = {x}")
for name, (mu, cov) in templates.items():
    ed = np.sqrt(((x - mu) ** 2).sum())
    md = np.sqrt((x - mu) @ np.linalg.inv(cov) @ (x - mu))
    print(f"  {name:5s}: Euclidean distance {ed:6.2f}   Mahalanobis distance {md:5.2f}")

# A case where the two distances disagree
x2 = np.array([137, 38.0])
print(f"\nTest person x2 = {x2} (between the two groups)")
res = {}
for name, (mu, cov) in templates.items():
    ed = np.sqrt(((x2 - mu) ** 2).sum())
    md = np.sqrt((x2 - mu) @ np.linalg.inv(cov) @ (x2 - mu))
    res[name] = (ed, md)
    print(f"  {name:5s}: Euclidean {ed:6.2f}   Mahalanobis {md:5.2f}")
print("  Euclidean picks:", min(res, key=lambda k: res[k][0]),
      "| Mahalanobis picks:", min(res, key=lambda k: res[k][1]))

Child: mean = [116.2   21.85], covariance =
[[161.96  60.7 ]
 [ 60.7   25.39]]

Adult: mean = [170.1   68.16], covariance =
[[131.21 131.76]
 [131.76 162.8 ]]

Test person x = [150.  47.]
  Child: Euclidean distance  42.13   Mahalanobis distance  8.13
  Adult: Euclidean distance  29.18   Mahalanobis distance  1.76

Test person x2 = [137.  38.] (between the two groups)
  Child: Euclidean  26.33   Mahalanobis  5.40
  Adult: Euclidean  44.78   Mahalanobis  2.94
  Euclidean picks: Child | Mahalanobis picks: Adult
